# Stress-Strain Profile - Computational Node

Demonstration of `StressStrainProfile` as a computational node with flexible state control.

Uses single `set_state()` method with clear parameter names matching structural engineering culture:
- **M-κ analysis**: `set_state(kappa, eps_bot)`
- **Unbalanced model**: `set_state(eps_top, eps_bot)`
- **Textbook RC design**: `set_state(eps_top, eps_s1, layer_index)`

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from bmcs_cross_section.matmod import EC2Concrete
from bmcs_cross_section.cs_design import (
    RectangularShape,
    BarReinforcement,
    ReinforcementLayout,
    CrossSection,
    StressStrainProfile
)
from bmcs_cross_section.cs_components import SteelRebarComponent

In [ ]:
# Construct cross-section
shape = RectangularShape(b=300, h=500)
concrete = EC2Concrete(f_ck=30)
steel_rebar = SteelRebarComponent(nominal_diameter=20, grade='B500B')

reinforcement = ReinforcementLayout()
c_nom = 50  # Concrete cover
# Bottom layer: 4 bars at z = c_nom
z_rebar_1 = c_nom
reinforcement.layers.append(
    BarReinforcement(z=z_rebar_1, component=steel_rebar, count=4)
)
# # Second layer: 4 bars at z = 2 × c_nom
# z_rebar_2 = 2 * c_nom
# reinforcement.layers.append(
#     BarReinforcement(z=z_rebar_2, component=steel_rebar, count=4)
# )

cs = CrossSection(shape=shape, concrete=concrete, reinforcement=reinforcement)

print(f"Cross-section: {cs.shape.b}×{cs.shape.h} mm")
print(f"Layer 1: 4ϕ{steel_rebar.nominal_diameter} at z={z_rebar_1:.0f} mm")
# print(f"Layer 2: 4ϕ{steel_rebar.nominal_diameter} at z={z_rebar_2:.0f} mm")
print(f"Total steel area: {cs.reinforcement.total_area:.0f} mm²")

## Example 1: M-κ Analysis (Iterative Curvature)

In [ ]:
# M-κ analysis: Iterate through curvature values
curvatures = np.linspace(0.000005, 0.000015, 6)
eps_bot = 0.002  # Fixed bottom strain

# Create profile once
profile = StressStrainProfile(cs)

results = []
for kappa in curvatures:
    # Update state for M-κ analysis mode
    profile.set_state(kappa=kappa, eps_bot=eps_bot)
    
    # Get force resultants
    F_c, F_s, N, M = profile.get_force_resultants()
    
    results.append({
        'kappa': kappa * 1000,  # ×10⁻³ mm⁻¹
        'eps_top': profile.eps_top * 1000,  # ‰
        'M': M / 1e6,  # kNm
        'N': N / 1000  # kN
    })

# Plot M-κ curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

kappa_plot = [r['kappa'] for r in results]
M_plot = [r['M'] for r in results]
eps_top_plot = [r['eps_top'] for r in results]

ax1.plot(kappa_plot, M_plot, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Curvature κ [×10⁻³ mm⁻¹]', fontsize=11)
ax1.set_ylabel('Moment M [kNm]', fontsize=11)
ax1.set_title('M-κ Relationship', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(kappa_plot, eps_top_plot, 's-', color='steelblue', linewidth=2, markersize=8)
ax2.axhline(y=-3.5, color='red', linestyle='--', label='εcu = -3.5‰ limit')
ax2.set_xlabel('Curvature κ [×10⁻³ mm⁻¹]', fontsize=11)
ax2.set_ylabel('Top strain εtop [‰]', fontsize=11)
ax2.set_title('Top Strain Evolution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nM-κ Analysis Results:")
print(f"  Curvature range: {curvatures[0]*1000:.3f} to {curvatures[-1]*1000:.3f} × 10⁻³ mm⁻¹")
print(f"  Moment range: {M_plot[0]:.2f} to {M_plot[-1]:.2f} kNm")

In [ ]:
# Unbalanced model: Fix top strain, vary bottom strain (teaching mode)
eps_top_fixed = -0.0035  # Concrete ultimate strain
eps_bot_values = np.linspace(0.001, 0.004, 5)

# Create profile once
profile = StressStrainProfile(cs)

results_unbalanced = []
for eps_bot in eps_bot_values:
    # Update state for unbalanced model mode
    profile.set_state(eps_top=eps_top_fixed, eps_bot=eps_bot)
    
    F_c, F_s, N, M = profile.get_force_resultants()
    
    results_unbalanced.append({
        'eps_bot': eps_bot * 1000,  # ‰
        'kappa': profile.kappa * 1000,  # ×10⁻³ mm⁻¹
        'F_c': F_c / 1000,  # kN
        'F_s': F_s / 1000,  # kN
        'imbalance': (F_c + F_s) / 1000  # kN
    })

# Plot force balance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

eps_b = [r['eps_bot'] for r in results_unbalanced]
F_c_plot = [r['F_c'] for r in results_unbalanced]
F_s_plot = [r['F_s'] for r in results_unbalanced]
imbalance = [r['imbalance'] for r in results_unbalanced]

ax1.plot(eps_b, F_c_plot, 'o-', label='Fc (compression)', linewidth=2, markersize=8)
ax1.plot(eps_b, F_s_plot, 's-', label='Fs (tension)', linewidth=2, markersize=8)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax1.set_xlabel('Bottom strain εbot [‰]', fontsize=11)
ax1.set_ylabel('Force [kN]', fontsize=11)
ax1.set_title('Force Components (εtop = -3.5‰ fixed)', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(eps_b, imbalance, 'D-', color='red', linewidth=2, markersize=8)
ax2.axhline(y=0, color='green', linestyle='--', linewidth=2, label='Equilibrium')
ax2.set_xlabel('Bottom strain εbot [‰]', fontsize=11)
ax2.set_ylabel('Force imbalance N [kN]', fontsize=11)
ax2.set_title('Equilibrium Check', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nUnbalanced Model Results:")
print(f"  Top strain (fixed): {eps_top_fixed*1000:.2f}‰")
print(f"  Bottom strain range: {eps_b[0]:.2f} to {eps_b[-1]:.2f}‰")
print(f"  Equilibrium at εbot ≈ {eps_b[np.argmin(np.abs(imbalance))]:.2f}‰")

## Example 3: Unbalanced Model (Top-Bottom Control)

In [ ]:
# Visualize a specific state
profile = StressStrainProfile(cs)
profile.set_state(eps_top=-0.0019, eps_s1=0.007)

fig, (ax_strain, ax_stress) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    sharey=True
)

profile.plot_stress_strain_profile(ax_strain, ax_stress, show_resultants=True)

plt.tight_layout()
plt.show()

## Example 2: Single State Visualization

## Summary

The `StressStrainProfile` is a computational node with a single `set_state()` method supporting multiple control modes:

```python
# M-κ analysis
profile.set_state(kappa=-0.00001, eps_bot=0.002)

# Textbook RC design  
profile.set_state(eps_top=-0.0035, eps_s1=0.005, layer_index=0)

# Unbalanced model
profile.set_state(eps_top=-0.0035, eps_bot=0.0078)
```

All kinematic transformations are handled internally using plane section assumption: **ε(z) = eps_bot - κ×z**

## Example 4: Textbook RC Design (Top + Reinforcement Layer)

In [ ]:
# Textbook RC design procedure:
# Fix εcu = -3.5‰ at top, iterate εs at bottom tension layer
eps_cu = -0.0035  # Concrete ultimate compressive strain
eps_s_values = np.linspace(0.001, 0.010, 7)  # Iterate steel strain

# Create profile once - single computational node
profile = StressStrainProfile(cs)

results_textbook = []
for eps_s in eps_s_values:
    # Update state using clean API (handles kinematics internally)
    profile.set_state(eps_top=eps_cu, eps_s1=eps_s, layer_index=0)
    
    # Get force resultants from current state
    F_c, F_s, N, M = profile.get_force_resultants()
    
    results_textbook.append({
        'eps_s': eps_s * 1000,  # ‰
        'kappa': profile.kappa * 1000,  # ×10⁻³ mm⁻¹
        'N': N / 1000,  # kN
        'M': M / 1e6,  # kNm
        'eps_bot': profile.eps_bottom * 1000  # ‰ (at z=0)
    })

# Plot equilibrium search
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

eps_s_plot = [r['eps_s'] for r in results_textbook]
N_plot = [r['N'] for r in results_textbook]
M_plot_tb = [r['M'] for r in results_textbook]

ax1.plot(eps_s_plot, N_plot, 'o-', linewidth=2, markersize=8, color='teal')
ax1.axhline(y=0, color='green', linestyle='--', linewidth=2, label='N=0 (equilibrium)')
ax1.set_xlabel('Steel strain εs [‰]', fontsize=11)
ax1.set_ylabel('Axial force N [kN]', fontsize=11)
ax1.set_title('Equilibrium Search (εcu=-3.5‰ fixed)', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(eps_s_plot, M_plot_tb, 's-', linewidth=2, markersize=8, color='darkorange')
ax2.set_xlabel('Steel strain εs [‰]', fontsize=11)
ax2.set_ylabel('Moment M [kNm]', fontsize=11)
ax2.set_title('Moment Capacity vs Steel Strain', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find equilibrium state (N ≈ 0)
N_abs = [abs(r['N']) for r in results_textbook]
equilibrium_idx = np.argmin(N_abs)
equilibrium_state = results_textbook[equilibrium_idx]

print(f"\nTextbook RC Design Results:")
print(f"  Fixed top strain: {eps_cu*1000:.2f}‰ (εcu)")
print(f"  Steel strain range: {eps_s_plot[0]:.2f} to {eps_s_plot[-1]:.2f}‰")
print(f"\n  Equilibrium found at:")
print(f"    εs = {equilibrium_state['eps_s']:.2f}‰")
print(f"    N = {equilibrium_state['N']:.2f} kN (≈0)")
print(f"    M = {equilibrium_state['M']:.2f} kNm")
print(f"    κ = {equilibrium_state['kappa']:.4f} × 10⁻³ mm⁻¹")

# Visualize the FINAL state (profile retains the last iteration state)
print(f"\nVisualizing final state (εs = {eps_s_values[-1]*1000:.2f}‰):")
fig, (ax_strain, ax_stress) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    sharey=True
)

profile.plot_stress_strain_profile(ax_strain, ax_stress, show_resultants=True)

plt.tight_layout()
plt.show()

In [ ]:
profile.plot_stress_strain_profile(ax_strain, ax_stress, show_resultants=True)
